In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, precision_score, recall_score, classification_report, f1_score
from sklearn.pipeline import FunctionTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as IMBPipeline
from pipelines import get_preprocessor, feature_engineer
import joblib
import os
import kagglehub

path = kagglehub.dataset_download("vinicius150987/titanic3")
excel_path = os.path.join(path, "titanic3.xls")
os.makedirs("images", exist_ok=True)
os.makedirs("models/", exist_ok=True)
os.makedirs("models/artifacts", exist_ok=True)

In [2]:
df = pd.read_excel(excel_path)
df = df.drop(columns=["boat", "body", "home.dest"])

X = df.drop(columns=["survived"])
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, shuffle=True, stratify=y)

joblib.dump(X_train, "models/artifacts/X_train.pkl")
joblib.dump(X_test, "models/artifacts/X_test.pkl")
joblib.dump(y_train, "models/artifacts/y_train.pkl")
joblib.dump(y_test, "models/artifacts/y_test.pkl")
print("X_train, X_test, y_train, y_test saved successfully")

X_train, X_test, y_train, y_test saved successfully


In [3]:
param_dist = [
    {
        "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
        "model__l1_ratio": [0, 0.5, 1],
        "model__solver": ["saga"],
        "model__class_weight": [None]
    }
]

In [4]:
pipe = IMBPipeline([
    ("feature_engineering", FunctionTransformer(feature_engineer, validate=False)),
    ("preprocessor", get_preprocessor()),
    ("smote", SMOTE(random_state=11)),
    ("model", LogisticRegression(max_iter=10000))
])

In [5]:
grid = GridSearchCV(pipe, param_grid=param_dist, cv=3, scoring="accuracy", verbose=1, n_jobs=-1)
grid.fit(X_train, y_train)

Fitting 3 folds for each of 18 candidates, totalling 54 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...iter=10000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model__C': [0.001, 0.01, ...], 'model__class_weight': [None], 'model__l1_ratio': [0, 0.5, ...], 'model__solver': ['saga']}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_au

In [6]:
y_pred = grid.predict(X_test)

In [7]:
confusion_matrix(y_test, y_pred)

array([[136,  26],
       [ 19,  81]])

In [8]:
grid.best_params_

{'model__C': 0.1,
 'model__class_weight': None,
 'model__l1_ratio': 0,
 'model__solver': 'saga'}

In [9]:
accuracy_score(y_test, y_pred)

0.8282442748091603

In [10]:
y_pred_proaba = grid.predict_proba(X_test)

In [11]:
roc_auc_score(y_test, y_pred_proaba[:, 1])

0.8785493827160494

In [12]:
precision_score(y_test, y_pred)

0.7570093457943925

In [13]:
recall_score(y_test, y_pred)

0.81

In [14]:
f1_score(y_test, y_pred)

0.782608695652174

In [15]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.84      0.86       162
           1       0.76      0.81      0.78       100

    accuracy                           0.83       262
   macro avg       0.82      0.82      0.82       262
weighted avg       0.83      0.83      0.83       262



In [16]:
grid.best_score_

np.float64(0.7841451766953199)

In [17]:
joblib.dump(grid, "models/logistic_regression_model.pkl")
print("Model saved")

Model saved


In [18]:
y_test.shape

(262,)